# Report fact-check — Sections 1–4

Every numeric claim in the draft report is recomputed here from the **data, the
checkpoints and the source**. Nothing is taken from code comments.

Each check prints `CLAIM`, the `MEASURED` value, and a verdict. `VERDICT: MISMATCH`
means the report must be corrected, not the notebook.

**Run order matters:** §1 establishes the authoritative station ordering that later
cells depend on.

**Memory:** §1 loads the full observation table (~several GB). Run on a machine with
enough RAM, or skip to §4 for checkpoint-only checks.


In [1]:
import os, sys, json, glob
import numpy as np
import pandas as pd

# --- EDIT THESE TWO PATHS IF NEEDED ---------------------------------------
PROJ = os.path.abspath(".")                     # the "Station MAE" directory
DATA = os.path.join(PROJ, "PeakWeatherDataset")  # or /home/renku/work/PeakWeatherDataset
# --------------------------------------------------------------------------
sys.path.insert(0, os.path.join(PROJ, "src"))

RESULTS = {}   # claim -> (claimed, measured, ok)

def check(name, claimed, measured, ok=None, note=""):
    if ok is None:
        ok = (str(claimed) == str(measured))
    RESULTS[name] = (claimed, measured, ok)
    tag = "OK" if ok else "MISMATCH"
    print(f"[{tag:8s}] {name}")
    print(f"           claim   : {claimed}")
    print(f"           measured: {measured}")
    if note:
        print(f"           note    : {note}")
    print()

print("PROJ:", PROJ)
print("DATA:", DATA, "exists:", os.path.isdir(DATA))

PROJ: /Users/aureliedejong/PycharmProjects/weather-station-model-with-transformers/Station MAE/notebooks
DATA: /Users/aureliedejong/PycharmProjects/weather-station-model-with-transformers/Station MAE/notebooks/PeakWeatherDataset exists: True


## 1. Authoritative station ordering

The model's station axis is `ds.stations_table.index` with `PFA` removed. Every
per-station result in the report is indexed by that order, so it must be read from
the library, not inferred.

This cell also writes `station_order.json` for reuse.

In [2]:
from peakweather.dataset import PeakWeatherDataset

ds = PeakWeatherDataset(
    root=DATA,
    parameters=["temperature","pressure","humidity",
                "wind_speed","wind_direction","precipitation"],
    compute_uv=True, station_type="meteo_station",
    imputation_method=None, freq="10min",
)
stations_table = ds.stations_table
order_all = stations_table.index.tolist()
STATION_ORDER = [s for s in order_all if s != "PFA"]

json.dump(STATION_ORDER, open("station_order.json", "w"))
check("Number of stations used", 155, len(STATION_ORDER))
print("first 10:", STATION_ORDER[:10])

ArrowInvalid: Could not open Parquet input source '<Buffer>': Parquet magic bytes not found in footer. Either the file is corrupted or this is not a parquet file.

## 2. Dataset facts (Section 3 of the report)

In [ ]:
raw = pd.read_parquet(os.path.join(DATA, "stations.parquet"))
check("PeakWeather total stations", 302, len(raw))
print(raw["station_type"].value_counts().to_string(), "\n")
check("meteo stations before filtering", 160,
      int((raw.station_type == "meteo_station").sum()))

h = stations_table.loc[STATION_ORDER, "station_height"]
check("Elevation range (m)", "203 to 3571",
      f"{h.min():.0f} to {h.max():.0f}",
      note=f"lowest {h.idxmin()}, highest {h.idxmax()}, median {h.median():.0f} m")

e = stations_table.loc[STATION_ORDER, "swiss_easting"]
n = stations_table.loc[STATION_ORDER, "swiss_northing"]
print(f"horizontal extent: {(e.max()-e.min())/1000:.0f} km E-W, "
      f"{(n.max()-n.min())/1000:.0f} km N-S")

In [ ]:
# Temporal coverage and sampling interval
first = pd.read_parquet(os.path.join(DATA, "observations", "2017.parquet"))
step = first.index.to_series().diff().dropna().mode()[0]
check("Sampling interval", "10 min", str(step))
check("Rows in 2017 (10-min grid)", 365*24*6, len(first))
years = sorted(int(os.path.basename(f)[:4])
               for f in glob.glob(os.path.join(DATA, "observations", "*.parquet")))
print("observation years available:", years)
last = pd.read_parquet(os.path.join(DATA, "observations", f"{years[-1]}.parquet"))
print(f"coverage: {first.index.min()}  ->  {last.index.max()}")
del first, last

In [ ]:
# Completeness over the TRAINING years only
CODE2VAR = {"tre200s0":"temperature", "prestas0":"pressure", "ure200s0":"humidity",
            "fkl010z0":"wind_speed",  "dkl010z0":"wind_dir",  "rre150z0":"precipitation"}
TRAIN_YEARS = [2017, 2018, 2019, 2020, 2021]
US = set(STATION_ORDER)

present = {v: 0 for v in CODE2VAR.values()}
total   = {v: 0 for v in CODE2VAR.values()}
paircnt = {}

for yr in TRAIN_YEARS:
    o = pd.read_parquet(os.path.join(DATA, "observations", f"{yr}.parquet"))
    T = len(o)
    seen = set()
    for (s, p) in o.columns:
        if s in US and p in CODE2VAR:
            v = CODE2VAR[p]
            k = int(o[(s, p)].notna().sum())
            present[v] += k; total[v] += T
            paircnt[(s, v)] = paircnt.get((s, v), 0) + k
            seen.add((s, v))
    for s in US:                       # columns absent entirely = all missing
        for v in CODE2VAR.values():
            if (s, v) not in seen:
                total[v] += T
                paircnt.setdefault((s, v), 0)
    del o

print(f"{'variable':15s}{'complete':>10s}")
for v in CODE2VAR.values():
    print(f"{v:15s}{100*present[v]/total[v]:>9.1f}%")

In [ ]:
# Sparse (station, variable) pairs -> these use the nearest-donor statistics
MIN_OBS = 50
sparse = sorted([(s, v, c) for (s, v), c in paircnt.items() if c < MIN_OBS])
check("Sparse-pair threshold MIN_OBS", 50, MIN_OBS)
print(f"sparse pairs (< {MIN_OBS} training obs): {len(sparse)}")
print(f"as fraction of {len(STATION_ORDER)}x6 pairs: "
      f"{100*len(sparse)/(len(STATION_ORDER)*6):.2f}%\n")
for s, v, c in sparse:
    print(f"  idx {STATION_ORDER.index(s):3d}  {s:6s} {v:14s} n={c}")

## 3. Which stations are the analysis outliers?

The results analysis flagged station indices **59** and **83** as having
targets that differ between the LSTM dump and the transformer dumps. This
cell names them and tests whether the sparse-statistics explanation holds.

In [ ]:
for i in [1, 16, 22, 55, 59, 83, 85, 114, 117, 128, 137, 146, 150]:
    s = STATION_ORDER[i]
    row = stations_table.loc[s]
    sp = [v for (ss, v), c in paircnt.items() if ss == s and c < MIN_OBS]
    print(f"idx {i:3d} = {s:6s} {str(row['station_name'])[:30]:30s} "
          f"{row['station_height']:6.0f} m   sparse: {sp if sp else '-'}")

In [ ]:
# Does the sparse explanation hold for 59 and 83?
for i in (59, 83):
    s = STATION_ORDER[i]
    sp = {v: c for (ss, v), c in paircnt.items() if ss == s}
    print(f"idx {i} = {s}: training counts {sp}")
    flagged = [v for v, c in sp.items() if c < MIN_OBS]
    print(f"   -> sparse variables: {flagged if flagged else 'NONE'}")
check("Stations 59/83 explained by sparse (donor) statistics",
      "yes (report claim)",
      "see counts above — verify before citing",
      ok=False,
      note="If neither is sparse, the LSTM/transformer target mismatch has "
           "another cause and the report text must be corrected.")

## 4. Model and configuration facts (Section 4)

Read from the checkpoints, so this section runs without the dataset.

In [ ]:
import torch
CKPT = os.path.join(PROJ, "checkpoints")

def load_cfg(run):
    p = os.path.join(CKPT, run, "best.ckpt")
    if not os.path.exists(p):
        return None, None, None
    c = torch.load(p, map_location="cpu", weights_only=False)
    sd = c["state_dict"]
    n = sum(v.numel() for v in sd.values() if hasattr(v, "numel"))
    return c.get("hyper_parameters", {}).get("cfg", {}), n, c

cfg27, n27, c27 = load_cfg("full_run_cloud_v27")
if cfg27 is None:
    print("v27 checkpoint not found — skipping section 4")
else:
    for k, claimed in [("d_model",384), ("enc_layers",8), ("dec_layers",2),
                       ("enc_heads",8), ("dec_heads",8), ("window",72),
                       ("temporal_patch",3), ("mask_ratio",0.5),
                       ("max_delta",36), ("delta_grid_stride",3),
                       ("residual_head",True), ("encoder_spatial_attn",True),
                       ("use_nll_loss",False), ("lr",1e-4), ("weight_decay",0.05)]:
        check(f"v27 cfg.{k}", claimed, cfg27.get(k, "<absent>"))
    check("v27 parameter count", "25.6 M", f"{n27/1e6:.2f} M",
          ok=abs(n27/1e6 - 25.63) < 0.05)
    check("v27 best epoch", 40, c27.get("epoch"))

In [ ]:
# Ablation arms and the parameter delta
rows = []
for run, label in [("full_run_cloud_v27","v27 spatial ON"),
                   ("full_run_cloud_v28","v28 spatial OFF"),
                   ("full_run_cloud_v29","v29 spatial OFF, MR0"),
                   ("lstm-baseline-v1",  "LSTM baseline")]:
    cfg, n, c = load_cfg(run)
    if cfg is None:
        print(f"  (missing: {run})"); continue
    rows.append(dict(run=label, params_M=round(n/1e6,2), epoch=c.get("epoch"),
                     spatial=cfg.get("encoder_spatial_attn","n/a"),
                     mask_ratio=cfg.get("mask_ratio","n/a")))
tbl = pd.DataFrame(rows); print(tbl.to_string(index=False), "\n")

cfg28, n28, _ = load_cfg("full_run_cloud_v28")
if cfg28 is not None:
    d = 384; per_block = 4*d*d + 4*d + 2*d      # MHA(in_proj+out_proj) + LayerNorm
    check("v27-v28 param gap = 8 spatial attention blocks",
          f"{8*per_block/1e6:.2f} M", f"{(n27-n28)/1e6:.2f} M",
          ok=abs((n27-n28) - 8*per_block) < 20_000)

In [ ]:
# Token counts quoted in Section 4.2
W, P, N = cfg27["window"], cfg27["temporal_patch"], 155
pos = W // P
vis = int(round(N * (1 - cfg27["mask_ratio"])))
check("Temporal positions per station", 24, pos)
check("Encoder tokens at mask 0.5", 1872, pos*vis, note=f"{pos} x {vis}")
check("Encoder tokens at mask 0.0", 3720, pos*N,  note=f"{pos} x {N}")
K = cfg27["max_delta"] // cfg27["delta_grid_stride"] + 1
check("Number of lead times K", 13, K)
print("lead times (h):", [round(k*cfg27["delta_grid_stride"]*10/60, 2) for k in range(K)])

In [ ]:
# Per-station normalisation statistics: shape, and the spread that motivates
# per-station inverse transform in the evaluation protocol
mean = np.array(cfg27["obs_stats_mean"]); std = np.array(cfg27["obs_stats_std"])
check("obs_stats shape", "(155, 6)", str(mean.shape))
names = ["temperature","pressure","humidity","wind_u","wind_v","precipitation"]
print(f"{'variable':15s}{'min std':>9s}{'median':>9s}{'max std':>9s}{'max/min':>9s}")
for v, nm in enumerate(names):
    c = std[:, v]
    print(f"{nm:15s}{c.min():9.3f}{np.median(c):9.3f}{c.max():9.3f}{c.max()/c.min():8.1f}x")
check("Wind std spread across stations", "14x",
      f"{std[:,3].max()/std[:,3].min():.1f}x",
      ok=abs(std[:,3].max()/std[:,3].min() - 14.0) < 1.0)

## 5. Test-set size and lead-time grid (from the prediction dumps)

In [ ]:
pred = os.path.join(PROJ, "test_results", "v27", "best_mr0.00", "predictions.pt")
if os.path.exists(pred):
    d = torch.load(pred, map_location="cpu", weights_only=False)
    check("Test windows", 11684, int(d["n_windows"]))
    check("Prediction tensor shape", "(M, 13, 155, 5)", str(tuple(d["preds"].shape)))
    grid = torch.unique(d["delta_steps"], dim=0)
    check("Delta grid uniform across windows", 1, grid.shape[0])
    print("delta steps:", grid[0].tolist())
    print("lead hours :", [round(x*10/60,2) for x in grid[0].tolist()])
    del d
else:
    print("predictions.pt not found — skipping")

## 6. Summary

In [ ]:
ok = sum(1 for _, _, o in RESULTS.values() if o)
print(f"{ok} / {len(RESULTS)} checks passed\n")
bad = [(k, c, m) for k, (c, m, o) in RESULTS.items() if not o]
if bad:
    print("REQUIRES ATTENTION — correct the report, not the notebook:")
    for k, c, m in bad:
        print(f"  - {k}\n      claimed : {c}\n      measured: {m}")
else:
    print("All claims verified.")